In [ ]:
import requests

ZONE_LOOKUP_PATH = "/Volumes/nyc_taxi/bronze/raw_data/taxi_zone_lookup.csv"
response = requests.get(
    "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
    timeout=30,
)
response.raise_for_status()

with open(ZONE_LOOKUP_PATH, "wb") as output_file:
    output_file.write(response.content)

print(f"Zone lookup saved at {ZONE_LOOKUP_PATH}")

In [ ]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

zone_lookup_schema = StructType(
    [
        StructField("LocationID", IntegerType(), nullable=False),
        StructField("Borough", StringType(), nullable=True),
        StructField("Zone", StringType(), nullable=True),
        StructField("service_zone", StringType(), nullable=True),
    ]
)

zone_lookup_df = (
    spark.read.option("header", True)
    .schema(zone_lookup_schema)
    .csv(ZONE_LOOKUP_PATH)
    .selectExpr(
        "CAST(LocationID AS INT) AS LocationID",
        "TRIM(Borough) AS Borough",
        "TRIM(Zone) AS Zone",
        "TRIM(service_zone) AS service_zone",
    )
)

(
    zone_lookup_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("nyc_taxi.bronze.zone_lookup")
)

print(f"Zone lookup rows written: {zone_lookup_df.count():,}")

In [ ]:
%sql

CREATE OR REPLACE TABLE nyc_taxi.silver.payment_type (paymentTypeID INT, label STRING);

INSERT INTO nyc_taxi.silver.payment_type
VALUES (1, 'Credit card'), (2, 'Cash'), (3, 'No charge'), (4, 'Dispute'), (5, 'Unknown'), (6, 'Voided trip');

CREATE OR REPLACE TABLE nyc_taxi.silver.trip_type (tripTypeID INT, label STRING);
INSERT INTO nyc_taxi.silver.trip_type
VALUES (1, 'Street-hail'), (2, 'Dispatch');

CREATE OR REPLACE TABLE nyc_taxi.silver.rate_code (rateCodeID INT, label STRING);
INSERT INTO nyc_taxi.silver.rate_code
VALUES (0, 'Unknown'), (1, 'Standard rate'), (2, 'JFK'), (3, 'Newark'), (4, 'Nassau or Westchester'), (5, 'Negotiated fare'), (6, 'Group ride');

CREATE OR REPLACE TABLE nyc_taxi.silver.vendor (vendorID INT, label STRING);
INSERT INTO nyc_taxi.silver.vendor
VALUES (1, 'Creative Mobile Technologies'), (2, 'VeriFone Inc.');